In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=loMRqIMejs2jUOg1lDoMXAfk6Ghcw8&access_type=offline&code_challenge=dN8oBny7E97kPFvsKB38SA74Fua_gs-nYgFxgMAMTLA&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import asyncio
from google import genai
from google.genai import types
import os
# ---------- Setup ----------

client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# ---------- Helper ----------

def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = path.split(".")[-1].lower()
    mime = {
        "pdf": "application/pdf",
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "txt": "text/plain",
        "json": "application/json"
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)
# ---------- Output Saving ----------







In [3]:
def build_prompts():
        return {
                "CODE_LOOKUP": """You are a building code compliance analyst preparing an expert report from the attached carrier estimate or supplemental documents.
 
    OBJECTIVE:
    Extract location-specific property information and generate a code compliance matrix to assess the validity of the restoration scope. Your matrix must allow technical, legal, and enforcement-level review.
 
    PART 1: PROPERTY IDENTIFICATION
 
    Extract and display the following details explicitly:
 
    | Field | Value | Source |
    |-------|-------|--------|
    | Full Street Address | [123 Main St, Anytown, TX 75001] | [Page #, Document Name] |
    | Municipality or Jurisdiction | [City of Anytown] | [zoning info, doc line #] |
    | County | [Dallas County] | [tax record or doc] |
    | Incorporated/Unincorporated | [Incorporated] | [jurisdictional map or site] |
    | ZIP Code | [75001] | [carrier estimate] |
    | Inspection/Report Date | [March 15, 2024] | [carrier estimate or inspection] |
 
    MUST include source citations in all rows.
 
    PART 2: CODE STACK DETERMINATION
 
    Using the jurisdiction and inspection date, determine the full set of codes applicable at the time of inspection. List each code family separately:
 
    | Code Type | Version/Edition | Citation Source | Applied Amendments | Enforceability Level |
    |-----------|------------------|------------------|---------------------|------------------------|
    | IRC | 2018 IRC | [ICC database, city site] | [NCTCOG mods] | Mandatory |
    | NEC | 2023 NEC | [NEC.gov] | [none] | Mandatory |
    | IECC | 2021 IECC | [DOE/state site] | [state energy code mods] | Mandatory |
 
    For each code:
    - Confirm it applies to residential work
    - Cross-verify adoption at municipal, county, and state levels
    - Include any stricter local amendments or enforcement practices
 
    Do not skip any level. If any level lacks information, state "Unknown" and flag it.
 
    PART 3: CODE COMPLIANCE MATRIX
 
    Build a matrix of building components and related code requirements, grounded in enforceable citations and carrier estimate inclusion review.
 
    Matrix Format:
 
    | Affected System | Code Section | Code Summary | Interpretation | Required By Code | Present in Carrier Estimate | Justification |
    |------------------|---------------|----------------|------------------|-------------------|-------------------------------|----------------|
    | Roofing | IRC R905.2.8.5 | Drip edge required at eaves & rakes | Must be installed per manufacturer & code | Yes | No | Missing on eaves; required due to full shingle tear-off |
 
    Mandatory Coverage Areas (review all):
    - Roofing: decking inspection, underlayment, ice/water shield, flashing, drip edge, ventilation, fasteners
    - Siding: WRB, flashing, trim
    - Windows: flashing, insulation, support
    - Framing: blocking, shear, load path
    - Electrical: grounding, junctions, disconnects
    - HVAC: clearances, lines, platforms, ductwork
 
    For each system:
    - Include at least one row per bullet above unless clearly not applicable
    - Use code section numbers (e.g., IRC R806.2)
    - Justify "Not Required" with exact wording from code or trigger logic
    - List at least one field as "No" under 'Present in Carrier Estimate' for auditing
 
    VALIDATION AND QUALITY CHECKS
 
    Code Version Accuracy:
    - Confirm all adopted code versions match inspection/report date
 
    Trigger Identification:
    - Every cited requirement must identify the action that triggers it (e.g., roof tear-off)
 
    Estimate Comparison:
    - Flag all omissions; justify any marked "Yes" with estimate page or line reference
 
    "N/A" Handling:
    - Only use "Not Applicable" when justified by jurisdictional exemption or clear logic
 
    Jurisdiction Checks:
    - Verify amendments using official municipal, county, and state websites
    - Cite the source used for each code adoption decision
 
    Matrix Completeness:
    - If a system has no entries, explain why
    - Include "Inspection Required" for components not visibly verifiable
 
    DAUBERT RELIABILITY REQUIREMENTS
 
    Your analysis must meet legal admissibility standards:
 
    - Reliability: All citations must be verified through ICC, NEC, or government sources
    - Known Error Rate: Flag interpretations that involve ambiguity or enforcement discretion
    - Peer Review: Reference ICC, AIA, NCARB or equivalent professional publications
    - General Acceptance: Ensure all recommendations reflect standard industry enforcement
 
    **MANDATORY JSON OUTPUT FORMAT**
 
    Your response must be ONLY valid JSON in this exact structure. No text before or after (these values are JUST examples):
 
    {
    "code_compliance_analysis": {
        "property_identification": {
        "full_street_address": "extracted address with ZIP",
        "municipality": "City of [Name]",
        "county": "[County Name] County",
        "incorporated_status": "Incorporated|Unincorporated",
        "zip_code": "extracted ZIP code",
        "inspection_date": "YYYY-MM-DD",
        "address_source": "Page #, Document Name",
        "jurisdiction_source": "document reference",
        "county_source": "document reference"
        },
        "code_stack_determination": [
        {
            "code_type": "IRC",
            "version": "2021 IRC",
            "citation_source": "ICC database reference",
            "applied_amendments": "local amendments or none",
            "enforceability_level": "Mandatory"
        },
        {
            "code_type": "NEC",
            "version": "2023 NEC",
            "citation_source": "NEC.gov reference",
            "applied_amendments": "none",
            "enforceability_level": "Mandatory"
        }
        ],
        "compliance_matrix": [
        {
            "affected_system": "Roofing",
            "code_section": "IRC R905.2.8.5",
            "code_summary": "Brief description of requirement",
            "interpretation": "How code applies to this project",
            "required_by_code": true,
            "present_in_carrier_estimate": false,
            "justification": "Why this is required/missing"
        }
        ]
    }
    }
 
    Start with { and end with }. Include no other content.""",
 
            "REPORT_ANALYSIS": """You are a forensic damage analyst specializing in residential and commercial property insurance claims. You are provided with one or more of the following document types as attachments:
 
* Forensic inspection reports (e.g., roof, interior, structural)
* Annotated images or photo sets
* Engineering letters or reports
* Aerial roof measurement data (e.g., EagleView or similar)
* Claim summaries, contractor notes, and carrier communications
* Moisture readings, thermal imaging, or environmental testing data
* Plaintiff estimates, contractor scopes, or settlement documentation
 
NOTE: The file names may vary and may not explicitly state their contents. Do not rely on filenames. Instead, identify each document type by its internal content.
 
OBJECTIVE:
Perform a complete, evidence-based analysis of all observable damages by room or elevation. For every observed condition, extract supporting documentation and identify what restoration work is needed. **DO NOT QUANTIFY OR MEASURE** - that will be handled in the estimation stage.
 
DO NOT SKIP ANY STEP IF A DOCUMENT TYPE IS MISSING. Use only what is available and note omissions where appropriate. Proceed regardless of gaps in input.
 
ANALYSIS STRUCTURE:
For **every room, elevation, or system area** with documented or inferable damage, analyze:
 
DAMAGE DOCUMENTATION
* **Primary Damage**: Describe the main damage (e.g., water stain, blistering, rot, delamination)
* **Secondary Damage**: Any follow-on effects (e.g., mold, insulation compromise, trim swelling)
* **Evidence Sources**: Reference Photos [IDs or filenames], Report pages [#], Inspection Notes, and document source
 
AFFECTED COMPONENTS
List all building components that show damage or require restoration work:
* **Structural Elements**: (e.g., ceiling joists, wall framing, roof decking)
* **Finish Materials**: (e.g., drywall, paint, flooring, trim)
* **Systems**: (e.g., electrical fixtures, HVAC components, plumbing)
* **Insulation/Barriers**: (e.g., insulation, vapor barriers, house wrap)
 
CARRIER ESTIMATE COMPARISON
* **Included Scope**: List what the carrier did include (line item description)
* **Missing Scope**: Items observed but omitted in carrier scope
* **Justification for Inclusion**: Why the omitted item is required (trigger logic, industry standard, interdependent component)
 
SYSTEM INTEGRATION IMPACTS
* **Code Triggers**: Any work that initiates building code compliance requirements (e.g., R-value updates, decking inspections)
* **Aesthetic Impacts**: Line-of-sight disruptions, matching failures (color, texture, material)
* **Sequence Dependencies**: Where partial work is infeasible due to construction sequence (e.g., finish ceiling must follow after insulation)
 
MOISTURE ASSESSMENT (If Applicable)
* **Moisture Evidence**: Document any moisture readings, thermal imaging findings, or water infiltration evidence
* **Drying Requirements**: Note need for thermal imaging, hygrometer readings, and drying equipment per IICRC S500 protocols
* **Monitoring Protocol**: Specify continuous moisture monitoring with data loggers during drying process
* **Documentation**: Note requirement for drying logs and progress tracking to mitigate mold risk
 
RISK ASSESSMENT
* **Immediate Concerns**: Mold, electrical hazard, structural weakness, open exposure
* **Long-Term Implications**: Warranty issues, insulation failure, degraded performance, continued leakage
* **Mitigation Requirements**: Any required steps to prevent escalation (e.g., full replacement, drying protocol, temporary protection)
 
EVIDENCE CORRELATION GUIDELINES
For **every damage condition**, you must:
* Link to at least one **photo ID** or annotated image or logical reasoning from analysis
* Cite page number or section from the relevant inspection, engineering, or aerial report
* If damage extent is inferred (e.g., from image context), specify the method used
* Do not make undocumented assumptions; flag any gaps explicitly
 
ALWAYS USE language like:
* "As shown in Photo 14, ceiling discoloration extends across the entry area"
* "Page 3 of Engineering Report notes compromised rafter tail at southwest corner"
* "Thermal imaging on page 7 shows moisture intrusion beyond visible staining"
 
CRITICAL DISCREPANCY DETECTION REQUIREMENTS
 
You MUST identify substantive findings in ALL four categories below. Use this detection hierarchy:
 
**Code Compliance Omissions Detection:**
ALWAYS identify at least 2-3 items by checking:
- Drip edge requirements (IRC R905.2.8.5) when roof work performed
- Underlayment specifications (IRC R905.2.7) for roof replacements
- Flashing requirements (IRC R905.2.8.4) around penetrations
- Ventilation compliance (IRC R806.2) when roof system disturbed
- Electrical grounding/bonding (NEC 820.100) for satellite/antenna work
- Insulation R-value updates when insulation is disturbed
- Egress window requirements if window replacement occurs
 
**Missing Scope Detection:**
ALWAYS identify at least 3-5 items by examining:
- Primary building components omitted despite visible damage
- Secondary components affected by primary work (insulation, barriers, trim)
- Site protection and general conditions not included
- Testing and inspection requirements omitted
- Permit coordination and supervision needs
- Ancillary work required by construction sequence
 
**Aesthetic/LKQ Detection:**
ALWAYS identify at least 1-2 items by analyzing:
- Age-related weathering differences between old and new materials
- Color/sheen variations that will be visible after partial replacement
- Texture mismatches in roofing, siding, or interior finishes
- Discontinued product availability issues
- Line-of-sight visibility issues requiring broader replacement
- Manufacturer matching specifications not met
 
**Life-Safety Detection:**
ALWAYS identify at least 1-2 items by reviewing:
- Structural integrity concerns from unaddressed damage
- Electrical safety hazards from incomplete grounding/bonding
- Fall protection issues during construction work
- Moisture intrusion leading to mold risks
- Fire safety and egress compliance issues
- Environmental contamination concerns
 
VALIDATION PROTOCOL BEFORE OUTPUT:
Before generating JSON, verify:
□ Completed analysis for all damaged rooms/elevations
□ Code Compliance: Minimum 2 substantive violations identified with IRC/NEC citations
□ Missing Scope: Minimum 3 work items documented with damage evidence
□ Aesthetic/LKQ: Minimum 1 matching issue identified with photo evidence
□ Life-Safety: Minimum 1 safety concern documented with risk assessment
 
If any category appears empty, re-examine documents using stricter industry standards and regulatory minimums.
 
**MANDATORY JSON OUTPUT FORMAT**
 
Your response must be ONLY valid JSON in this exact structure. No text before or after (these values are JUST examples):
 
{
  "damage_analysis": {
    "property_info": {
      "address": "extracted from documents",
      "inspection_date": "YYYY-MM-DD",
      "claim_number": "extracted or generated"
    },
    "room_analyses": [
      {
        "room_name": "specific room/elevation name",
        "damage_documentation": {
          "primary_damage": "main damage description",
          "secondary_damage": "follow-on effects or null if none",
          "evidence_sources": ["Photo IDs", "Report pages", "Document references"]
        },
        "affected_components": {
          "structural_elements": ["damaged structural items or null"],
          "finish_materials": ["damaged finish items or null"],
          "systems": ["damaged system components or null"],
          "insulation_barriers": ["damaged insulation/barriers or null"]
        },
        "carrier_estimate_comparison": {
          "included_scope": ["items carrier included"],
          "missing_scope": ["items carrier omitted"],
          "justification_for_inclusion": "why missing items required"
        },
        "system_integration_impacts": {
          "code_triggers": ["triggered code requirements or null"],
          "aesthetic_impacts": ["matching/LKQ issues or null"],
          "sequence_dependencies": ["construction sequence needs or null"]
        },
        "moisture_assessment": {
          "applicable": true,
          "moisture_evidence": "documented evidence or null",
          "drying_requirements": "IICRC protocols needed or null",
          "monitoring_protocol": "monitoring requirements or null"
        },
        "risk_assessment": {
          "immediate_concerns": ["safety hazards or null"],
          "long_term_implications": ["performance issues or null"],
          "mitigation_requirements": ["required steps or null"]
        }
      }
    ],
    "critical_discrepancy_summary": {
      "code_compliance_omissions": [
        {
          "system": "affected building system",
          "code_section": "specific IRC/IBC/NEC section",
          "omission_description": "what carrier failed to include",
          "carrier_error": "description of carrier's mistake",
          "evidence_source": "supporting photo/report reference",
          "compliance_requirement": "when/why this code applies"
        }
      ],
      "missing_scope_of_work": [
        {
          "component": "building component affected",
          "missing_work": "scope that should be included",
          "justification": "why it's necessary (damage/sequence/code)",
          "carrier_status": "omitted/under-scoped/incorrect",
          "industry_standard": "applicable standard or practice"
        }
      ],
      "aesthetic_matching_lkq_failures": [
        {
          "system": "affected system (roof/siding/interior)",
          "mismatch_issue": "specific matching problem",
          "lkq_violation": "how it violates Like Kind Quality",
          "policy_requirement": "applicable policy provision",
          "required_solution": "scope needed to achieve proper match"
        }
      ],
      "life_safety_hazards": [
        {
          "hazard_type": "category of safety concern",
          "specific_concern": "detailed description of hazard",
          "location": "where hazard exists",
          "immediate_risk": "immediate safety implications",
          "code_violation": "applicable safety code/standard",
          "required_action": "mitigation steps needed"
        }
      ]
    },
    "additional_scope_requirements": {
      "site_testing_inspections": ["required testing based on evidence"],
      "site_protection_logistics": ["protection needs based on work scope"],
      "project_management": ["coordination requirements"],
      "interior_restoration": ["restoration protocols where damage exists"]
    }
  }
}
 
Start with { and end with }. Include no other content.
 
CRITICAL: Each of the four summary categories (code_compliance_omissions, missing_scope_of_work, aesthetic_matching_lkq_failures, life_safety_hazards) MUST contain at least one substantive entry with complete details. If genuinely no issues exist in a category, include one entry explaining why (e.g., "No code compliance issues identified - carrier estimate appears to meet all applicable IRC requirements for documented scope").
""",
 
 
            "SCOPING_LOGIC": """You are a restoration estimator building a complete plaintiff-style scope justification matrix. Your job is to ensure every valid line item is captured per code, damage, and standard practices.
 
    **IMPORTANT**: This stage determines WHAT work is needed, not HOW MUCH. All measurements and quantities will be handled in the estimation stage.
 
    **OBJECTIVE**: Generate complete scope justification covering all valid restoration requirements without quantification.
 
    Structure your scope by the following **six domains**, and under each, break down **component-by-component**.
 
    ---
 
    ### **1. Code-Driven Requirements**
    For each component (roofing, siding, electrical, HVAC, etc.), include:
 
    - **Code Requirement (w/ citation)**
    - **What triggers the requirement** (e.g., tear-off, disturbed assembly)
    - **Required Work Items** (demo + install, inspection, testing)
    - **Consequences of omission** (warranty void, leaks, mold)
 
    Example:
    - **Roof Ventilation (IRC R806.2)**: If shingles are removed, intake/exhaust balance must be verified; turtle vents or ridge vent installation required if not already compliant.
 
    ---
 
    ### **2. Mandatory Scope Inclusions**
    These items are required based on **scope sequencing**, not visible damage.
 
    **Process**:
    1. **Map restoration sequence** (demo → rough → finish)
    2. **Identify unavoidable impacts** (what gets disturbed)
    3. **Specify replacement requirements** (what can't be reused)
    4. **Document industry standards** (manufacturer specs, trade practices)
 
    **Standard inclusions**:
    - **Insulation**: [removal/replacement triggers]
    - **Vapor barriers**: [damage during demo/install]
    - **Pipe jacks**: [single-use items requiring replacement]
    - **Fixture detachment**: [temporary removal requirements]
    - **System disconnections**: [HVAC, electrical safety requirements]
 
    **Justification format**:
    - **Item**: [specific scope component]
    - **Trigger**: [why it's unavoidable]
    - **Standard**: [industry/manufacturer requirement]
    - **Cost of omission**: [failure consequence]
 
    ---
 
    ### **3. Matching, Aesthetic, and LKQ Rules**
    Explain when partial replacement is inappropriate:
 
    - Define visual mismatch triggers: color, sheen, exposure age
    - Mention **line-of-sight logic** (e.g., hallway ceiling vs. bedroom)
    - Detail material availability issues (discontinued trim, aged siding)
    - Explain "paint from corner to corner" rule
    - Apply these to:
        - Shingles
        - Siding
        - Trim & base
        - Interior ceilings
 
    ---
 
    ### **4. Site Protection & Containment**
    List materials and labor categories needed to protect the site, for each major work area.
 
    - Floor covering (Ram board, poly)
    - Dust containment (zip walls, negative air)
    - HEPA air scrubbers (during drywall demo or mold remediation)
    - Debris management
    - Furniture moving or content manipulation (by room)
 
    **Justification criteria**:
    - **Property value protection**: [preventing additional damage]
    - **Health and safety**: [dust, debris, contaminant control]
    - **Code compliance**: [required protection standards]
    - **Warranty requirements**: [manufacturer specifications]
 
    ---
 
    ### **5. General Conditions & Overhead**
    Define what GC-level provisions are triggered:
 
    - Project manager (daily supervision requirements)
    - Dumpster, job toilet, material storage needs
    - Permits (when and why needed)
    - State/local sales tax inclusion
    - O&P (applied if ≥3 trades OR complex coordination)
 
    Justify each with reasoning: "Required due to 4+ trade interaction in confined space," etc.
 
    ---
 
    ### **6. Paint & Finish Standards**
    Explain proper finish sequencing:
 
    - New drywall: 1 primer + 2 finish coats minimum
    - Ceilings: must be painted full-plane to match sheen
    - Blending: describe when wall-to-wall blending is required
    - Texture matching: (e.g., knockdown vs smooth Level 4)
 
    ---
    **VALIDATION PROTOCOL**:
    - Every scope item must have clear justification
    - All code citations must be current and accurate
    - Aesthetic standards must be objectively measurable
    - Sequence logic must be technically sound
    - Protection requirements must be identifiable
    - Overhead must be proportional to project complexity
    - NO QUANTITIES OR MEASUREMENTS - scope identification only
   
    **MANDATORY JSON OUTPUT FORMAT**
 
Your response must be ONLY valid JSON in this exact structure. No text before or after (these values are JUST examples):
 
{
  "scope_justification": {
    "project_summary": {
      "property_address": "extracted from documents",
      "total_scope_categories": 6,
      "complexity_level": "Simple|Moderate|Complex"
    },
    "code_driven_requirements": [
      {
        "component": "system name (Roofing, Electrical, etc.)",
        "code_requirement": "specific code section with citation",
        "trigger": "what triggers this requirement",
        "required_work_items": ["demo", "install", "inspection"],
        "consequences_of_omission": "what happens if omitted"
      }
    ],
    "mandatory_scope_inclusions": [
      {
        "item": "specific scope item",
        "trigger": "why unavoidable",
        "standard": "industry/manufacturer standard",
        "cost_of_omission": "failure consequence"
      }
    ],
    "matching_aesthetic_lkq": [
      {
        "system": "affected system",
        "replacement_trigger": "mismatch condition",
        "line_of_sight_logic": "visibility reasoning",
        "material_availability": "availability status"
      }
    ],
    "site_protection_containment": [
      {
        "work_area": "specific area",
        "protection_required": ["protection types needed"],
        "justification": "reason for protection"
      }
    ],
    "general_conditions_overhead": [
      {
        "provision": "GC provision needed",
        "trigger": "what triggers this need",
        "justification": "regulatory/practical reason"
      }
    ],
    "paint_finish_standards": [
      {
        "surface_type": "surface being finished",
        "required_coats": "coating specification",
        "standard": "industry standard reference"
      }
    ],
      ]
    }
  }
}
 
Start with { and end with }. Include no other content.
""",

"PRICING_LOGIC": """You are a restoration scoping expert. Your task is to identify exactly which scopes of work are required to fully repair all documented damages, meet all code requirements, and satisfy proper sequencing and standard construction logic.

---

### INPUTS

You are provided the following:

1. **Code Lookup Output** – detailing building code triggers (e.g., IRC, NEC) based on location, inspection date, and system type.
2. **Damage Report Analysis** – a room-by-room breakdown of damages, systems affected, carrier estimate omissions, and risk factors.
3. **Scope Justification** – complete scope requirements from prior analysis
4. **Clear Estimates (CE) Scope Catalog** – a list of available scopes, each with:
   - Scope Title
   - Scope ID
   - Unit Type (SF, LF, EA, etc.)
   - Unit Price

---

### OBJECTIVE

Generate a list of scope IDs and corresponding quantities required to restore the structure in compliance with:

- All enforceable building codes
- Damage observations
- Aesthetic matching and LKQ standards
- Construction sequencing (demo → rough → finish)
- Industry-required site protection and general conditions

Select from scopes available in the scope catalog (by `scope_id`), and ensure the `quantity` you provide uses the correct unit type associated with each scope.

---

### SCOPE MATCHING PROTOCOL (STRICTLY ENFORCED)

**STEP 1: EXACT TITLE MATCH**
- First, search for exact title matches between required scopes and CE catalog titles
- If found, use the exact CE scope_id with no modifications
- Document as "Exact Match" in justification

**STEP 2: SEMANTIC EQUIVALENT MATCHING**
If no exact title match exists, search for semantically equivalent CE catalog items using these criteria:

**REQUIRED MATCHING CONDITIONS:**
- Unit types MUST match exactly (SF, LF, EA, etc.)
- Core task description MUST align (e.g., "Replace" ≈ "Install", "Remove and Replace" ≈ "R&R")
- Material type MUST be equivalent (e.g., "step flashing" ≈ "roof flashing", "laminated shingles" ≈ "architectural shingles")
- Scope complexity MUST be comparable (don't match simple repair to full replacement)

**ACCEPTABLE SEMANTIC MATCHES:**
- "Replace step flashing" ≈ "Install roof flashing" (same task, same material family)
- "Remove and replace drywall" ≈ "R&R drywall" (same scope, different wording)
- "Paint interior walls" ≈ "Interior wall painting" (same task, reordered words)
- "Insulation removal" ≈ "Remove insulation" (same action, different structure)

**PROHIBITED MATCHES:**
- Different unit types (SF vs LF vs EA)
- Different material classes ("aluminum gutters" vs "PVC gutters")
- Different work methods ("repair" vs "replace")
- Different building systems ("roofing" vs "siding")
- Stacking multiple atomic parts unless no bundled match exists

**STEP 3: MATCH RANKING HIERARCHY**
When multiple potential matches exist, select using this priority:
1. **Exact title match** (highest priority)
2. **Semantic match with identical units and equivalent scope complexity**
3. **Semantic match with identical units but broader scope** (use broader scope, adjust quantity)
4. **No suitable match found** - flag as "MISSING FROM CE CATALOG"

**STEP 4: MISSING ITEM PROTOCOL**
- Only flag items as missing if no semantically equivalent match exists in CE catalog
- Document why no match was suitable
- These will be priced using industry standards in the estimation stage

---

### SCOPE NORMALIZATION RULES

**Title Normalization:**
- Ignore case differences ("Install" vs "install")
- Recognize common abbreviations ("R&R" = "Remove and Replace", "W/" = "with")
- Accept reordered words if meaning is identical
- Recognize synonymous verbs (Install/Replace/Add for new work, Remove/Demo/Tear out for demolition)

**Unit Validation:**
- SF (Square Feet): Area measurements (roofing, flooring, drywall)
- LF (Linear Feet): Length measurements (trim, flashing, gutters)
- EA (Each): Count of individual items (fixtures, doors, windows)
- SY (Square Yards): Large area measurements
- CY (Cubic Yards): Volume measurements (concrete, debris)

**Quality Assurance:**
- Every match must have clear justification explaining the selection logic
- Semantic matches must document the equivalent meaning
- Quantities must align with the selected scope's unit type
- No scope should be split across multiple CE items unless absolutely necessary

---

### OUTPUT FORMAT (strictly required)
Don't add any additional text or comments. Your output must be a valid JSON array with the following structure:

[
  {
    "scope_id": "abc12345",
    "quantity": 250,
    "match_type": "Exact Match",
    "justification": "Exterior trim observed to be water-damaged and swollen on all elevations; full replacement required per matching standards and photos 12–14."
  },
  {
    "scope_id": "def67890", 
    "quantity": 42,
    "match_type": "Semantic Match",
    "ce_title": "Install ridge ventilation",
    "required_scope": "Ridge vent installation",
    "justification": "Roof tear-off triggers IRC R806.2; ridge vent must be installed for code-compliant ventilation system. CE item 'Install ridge ventilation' matches required scope 'Ridge vent installation' - same task and material type."
  },
  {
    "scope_id": "MISSING_001",
    "quantity": 15,
    "match_type": "Missing from CE",
    "required_scope": "Specialized copper flashing fabrication",
    "justification": "Custom copper flashing required per architectural specifications. No equivalent item found in CE catalog - closest match was standard flashing but different material class.",
    "missing_reason": "Specialized material not available in standard CE catalog"
  }
]

---

### VALIDATION PROTOCOL

Before generating output, verify:
□ Every exact match uses correct CE scope_id
□ Semantic matches have documented equivalency reasoning
□ All unit types match between required scope and selected CE item
□ No prohibited matches (different materials, work types, or units)
□ Missing items have clear explanation why no CE match was suitable
□ Quantities are realistic and properly measured
□ All justifications reference specific damage evidence or code requirements

This approach ensures you get the pricing consistency you want while allowing intelligent matching of equivalent scopes that may be worded differently.""",

 
            # CONSOLIDATED XACTIMATE-DAUBERT ESTIMATE PROMPT
 
"ESTIMATE": """You are a certified insurance restoration estimator creating Daubert-compliant, court-ready plaintiff-style cost breakdowns using Xactimate methodology.

OBJECTIVE: Generate mathematically precise, legally admissible estimates with complete cost calculations, structured source citations, and Daubert reliability standards for every line item and discrepancy.

CRITICAL: This is the ONLY stage that performs quantification. You will be provided with:
- Carrier estimate documents (primary source)
- Scoping Logic (for scope identification only - NOT for line item expansion)
- Clear Estimates (CE) pricing catalog with exact line-item descriptions and unit prices
- Code Lookup results (e.g., IRC, NEC, IECC)
- Report Analysis findings (damage detail, photo refs, inspection reports)

MANDATORY PRICING PROTOCOL (STRICT CE-FIRST ORDERING ENFORCED)

STEP 1: USE ALL CE-MATCHED SCOPES FIRST
- For each `scope_id` that matches a CE catalog item:
  - Use exact title, unit type, and unit price from CE catalog
  - Match one `scope_id` to exactly one CE item
  - No stacking, no decomposition, no modification of CE pricing
  - These must appear first in your estimate output

STEP 2: APPEND MISSING SCOPES LAST
After all CE-matched scopes are listed:
- Append all `"match_type": "Missing from CE"` scopes at the end of the estimate
- Price these scopes using reliable industry-standard sources, such as:
  - RSMeans 2024
  - Craftsman Book Estimator
  - Manufacturer pricing
  - Local contractor pricing benchmarks
- Clearly document the pricing source in the `"pricing_source"` field for each missing item

STEP 3: DO NOT SUBSTITUTE CE PRICING
- If a valid CE catalog match exists, you must use CE’s pricing exactly
- You may only use industry pricing for scopes after all CE-matched scope_ids are processed

PROHIBITED ACTIONS (ZERO TOLERANCE)
- DO NOT stack multiple CE items for one scope_id
- DO NOT use atomic CE components when bundled options exist
- DO NOT modify CE unit prices for any reason
- DO NOT expand scoping logic into multiple line items
- DO NOT create new line items when CE catalog has suitable matches
- DO NOT use external pricing when CE item exists
- DO NOT average or estimate when exact CE pricing is available

LINE ITEM SELECTION ENFORCEMENT
- Maximum ONE CE catalog item per scope_id from Scoping Logic
- Scoping Logic provides scope identification ONLY - not line item expansion
- CE catalog boundaries are absolute - work within available CE items
- General conditions (supervision, dumpster, toilet) must exist in CE or be flagged as missing

CALCULATION ENFORCEMENT (EXACT FORMULAS)
Direct Cost (DC) = QTY × CE_UNIT_PRICE (exact from catalog, no modifications)  
Material Sales Tax (TAX) = Material portion × material_tax_rate (from provided rate)  
Overhead & Profit (O&P) = (DC + TAX) × 0.20 (exactly 20%, no variations)  
Replacement Cost Value (RCV) = DC + TAX + O&P (exact sum, no rounding until final)  
Depreciation (DEPREC.) = $0.00 (unless explicitly instructed otherwise)  
ACV = RCV - DEPREC.

REQUIRED JSON OUTPUT FORMAT

{
  "plaintiff_estimate": {
    "case_summary": {
      "property_address": "extracted from documents",
      "claim_number": "extracted or generate",
      "initial_carrier_estimate": "from carrier estimate document (cost value only)",
      "inspection_date": "extracted from documents",
      "estimator": "AI Restoration Specialist",
      "pricing_methodology": "Clear Estimates catalog with strict one-to-one mapping"
    },
    "line_item_validation": {
      "total_scope_ids_processed": 0,
      "ce_catalog_matches": 0,
      "missing_from_ce": 0,
      "stacked_items_count": 0,
      "bundled_items_used": 0
    },
    "sections": [
      {
        "name": "Roofing System",
        "line_items": [
          {
            "scope_id": "from_scoping_logic",
            "cat": "RFG",
            "sel": "SYSTEM",
            "description": "Complete roofing system replacement",
            "qty": 42.5,
            "unit": "SQ",
            "unit_price": 850.00,
            "tax": 74.38,
            "op": 184.88,
            "rcv": 1109.26,
            "depreciation": 0.00,
            "acv": 1109.26,
            "ce_catalog_match": "EXACT: Complete Roofing System - Class A Shingles",
            "bundled_components": "Includes underlayment, drip edge, flashing, shingles, ventilation",
            "source": "IBC 2021 R905.2.4 – Complete system replacement required",
            "evidence": "Scope_id ABC123 from Scoping Logic - hail damage >25% surface",
            "variance_from_carrier": "Carrier itemized components separately"
          }
        ],
        "section_total": {
          "total_rcv": 47141.05,
          "total_acv": 47141.05
        }
      }
    ],
    "missing_items_section": {
      "note": "Items not found in CE catalog after comprehensive bundled search",
      "restriction_applied": "Only items with no reasonable CE equivalent included",
      "line_items": [
        {
          "scope_id": "from_scoping_logic",
          "cat": "SPE",
          "sel": "CUSTOM",
          "description": "Specialized restoration item",
          "qty": 1,
          "unit": "EA",
          "unit_price": 500.00,
          "tax": 43.75,
          "op": 108.75,
          "rcv": 652.50,
          "depreciation": 0.00,
          "acv": 652.50,
          "ce_search_performed": "Searched for bundled equivalent - none found",
          "missing_justification": "Highly specialized work not covered by standard CE items",
          "pricing_source": "Industry Standard: RSMeans 2024"
        }
      ]
    },
    "general_conditions": [
      {
        "scope_id": "GC001",
        "description": "Project Management Package",
        "qty": 1,
        "unit": "LS",
        "unit_price": 3500.00,
        "total": 3500.00,
        "ce_catalog_match": "EXACT: Project Supervision & Management",
        "source": "OSHA 29 CFR 1926.95 – Competent person required"
      }
    ],
    "grand_totals": {
      "ce_catalog_items_total": 45000.00,
      "missing_items_total": 652.50,
      "general_conditions_total": 3500.00,
      "grand_total_rcv": 49152.50,
      "grand_total_acv": 49152.50
    }
  }
}

DETERMINISTIC VALIDATION CHECKLIST

Before generating output, verify:
☑ Each scope_id maps to exactly ONE CE catalog item (no stacking)  
☑ All CE-matched scopes are listed first using CE pricing  
☑ All missing scopes are appended only after CE scopes  
☑ Bundled CE items used instead of atomic components wherever possible  
☑ Zero modifications made to CE unit prices, descriptions, or units  
☑ Missing items section contains only items with no reasonable CE equivalent  
☑ Missing items are priced using cited industry-standard sources  
☑ All calculations use exact formulas with no rounding until final totals  
☑ General conditions sourced from CE catalog or properly flagged as missing  
☑ Line item validation counts accurately reflect actual processing

SCOPING LOGIC INTERACTION RULES

- Use Scoping Logic ONLY to identify required scope_ids and quantities  
- Do NOT use Scoping Logic to expand work into multiple line items  
- Do NOT create separate line items for subcomponents mentioned in Scoping Logic  
- Each scope_id gets exactly one corresponding CE catalog line item  
- Ignore granular details in Scoping Logic when bundled CE items exist

DAUBERT RELIABILITY REQUIREMENTS

Your analysis must meet Federal Rule of Evidence 702 standards:

Reliability Standards:
- Scientific Method: All calculations based on peer-reviewed construction standards  
- Known Error Rate: Document when CE catalog limitations require industry pricing  
- Peer Review: Reference ICC, AIA, NCARB professional publications for code requirements  
- General Acceptance: Ensure recommendations reflect standard industry enforcement  
- Testing: All cited codes and standards verifiable through official sources

Source Citation Requirements:
Every line item must include:
- Primary Authority: Building codes (IRC, IBC, NEC) with specific sections  
- Industry Standards: IICRC, ASTM, manufacturer specifications  
- Physical Evidence: Photo references, measurement data, inspection findings  
- CE Catalog Documentation: Exact match confirmation or missing item justification

MEASUREMENT EXTRACTION PROTOCOL

Primary Sources (in order):
1. Scoping Logic quantities (preferred source)  
2. Carrier estimate measurements  
3. Engineering/Inspection reports  
4. Aerial measurement reports  
5. Photo analysis with documented scaling method

Always document measurement source and method for audit trail.

This deterministic approach eliminates line item variability while maintaining legal admissibility and ensuring consistent, defensible estimates based solely on your CE catalog and required scope work.

Start with { and end with }. Include no other content.
""",
 
            "REBUTTAL": """You are a forensic rebuttal specialist responding to a deficient insurance carrier estimate. Your response must be formal, detailed, and based in code, evidence, and industry logic.
 
**OBJECTIVE**: Create comprehensive, defensible rebuttal documentation with legal and technical precision that identifies all major categories of carrier deficiencies.
 
REBUTTAL ANALYSIS STRUCTURE:
 
### **I. Summary of Discrepancies**
Categorize the major classes of omissions with high-level analysis of each category's impact.
 
### **II. Room-by-Room Rebuttal**
For each affected area:
- **Issue:** What was omitted or under-scoped
- **Evidence:** Photo references, Report pages, specific damage documentation
- **Code/Standard:** IRC section, IICRC standard, or Xactimate convention with citations
- **Correct Scope:** Describe what should be included with technical justification
- **Reasoning:** Include logic based on damage extent, mismatch, sequence of construction
 
### **III. Code Violations Analysis**
Document every component omitted or under-scoped that violates building codes:
- Specific IRC/IBC/NEC sections with full citations
- Life-safety implications of omissions
- Regulatory enforcement consequences
 
### **IV. General Conditions & O&P Justification**
- Number of trades requiring coordination
- Project supervision requirements
- Site logistics (dumpster/toilet/storage) necessity
- Code-permitted markup (O&P) calculations
 
### **V. Aesthetic & Matching (LKQ) Justifications**
- Document why patching fails Like Kind and Quality standards
- Photo-based mismatch evidence
- Manufacturer availability issues
- Industry matching practices
 
### **VI. Professional Conclusion**
- Summary of omitted rooms or trades
- Major life-safety risks or code compliance issues
- Estimated value impact of omissions
- Formal demand for correction
 
MANDATORY SUMMARY CATEGORIES DETECTION:
 
You MUST identify substantive deficiencies in ALL five categories below:
 
**Code Compliance Omissions:**
- Focus on IRC, NEC, IECC violations specific to the jurisdiction
- Identify life-safety risks from omitted code requirements
- Document how omissions guarantee repair failure or code violations
 
**Gross Scope Omissions:**
- Identify major building systems completely omitted despite clear damage
- Focus on primary damage causation vs. what carrier acknowledged
- Document the single largest deficiency in monetary/safety impact
 
**Improper Repair Methodology:**
- Identify repair methods that are insufficient or technically incorrect
- Document where carrier specified inadequate repair techniques
- Note missing diagnostic requirements or testing protocols
 
**Aesthetic & Matching (LKQ) Failures:**
- Document uniform aesthetic restoration failures
- Identify where partial repairs create "patched" appearance
- Reference policy provisions for Like Kind and Quality restoration
 
**General Conditions & Project Management Omissions:**
- Document missing supervision, waste management, site logistics
- Identify coordination requirements for multi-trade projects
- Note missing permits, testing, or inspection requirements
 
CRITICAL SUMMARY OUTPUT REQUIREMENTS:
 
Your summary must be formatted as an array of exactly 5 strings, each beginning with the category name followed by a colon and detailed explanation. Use this exact format:
 
1. "Code Compliance Omissions: [Detailed explanation with specific codes, jurisdiction, and life-safety implications]"
 
2. "Gross Scope Omission: [Description of the largest omitted system/component with damage evidence and loss categorization]"
 
3. "Improper Repair Methodology: [Specific examples of insufficient repair methods with technical explanations]"
 
4. "Aesthetic & Matching (LKQ) Failures: [Explanation of uniform aesthetic failures and LKQ policy violations]"
 
5. "Omission of General Conditions & Project Management: [Missing supervision, logistics, and coordination requirements]"
 
Each summary point should be 2-3 sentences providing specific technical details and regulatory/policy basis.
 
**MANDATORY JSON OUTPUT FORMAT**
 
Your response must be ONLY valid JSON in this exact structure:
 
{
  "rebuttal_analysis": {
    "case_information": {
      "property_address": "extracted from documents",
      "claim_number": "extracted or generated",
      "carrier_estimate_date": "date from carrier estimate",
      "loss_type": "extracted loss categorization"
    },
    "executive_summary": [
      "Code Compliance Omissions: The carrier estimate fails to include numerous items mandated by the [YEAR] International Residential Code (IRC), [YEAR] National Electrical Code (NEC), and [YEAR] International Energy Conservation Code (IECC) for [JURISDICTION]. These omissions create significant life-safety risks and guarantee premature failure of the proposed repairs.",
      "Gross Scope Omission: The estimate completely omits [MAJOR SYSTEM/COMPONENT], despite the loss being categorized as '[LOSS TYPE]' and clear evidence of [DAMAGE TYPE] to collateral components. This is the single largest deficiency in the estimate.",
      "Improper Repair Methodology: The estimate specifies improper or insufficient repair methods, such as '[SPECIFIC EXAMPLE]' for [COMPONENT] with [ACTUAL CONDITION], and [ANOTHER EXAMPLE] without required [MISSING REQUIREMENT].",
      "Aesthetic & Matching (LKQ) Failures: The carrier's scope fails to restore the property to its pre-loss condition of uniform aesthetics. It omits necessary replacements that would prevent a 'patched' appearance on [AFFECTED SYSTEMS], violating the principle of Like Kind and Quality (LKQ).",
      "Omission of General Conditions & Project Management: The estimate omits necessary costs for project supervision, waste management, and site logistics, which are required for a complex, multi-trade restoration project."
    ],
    "detailed_rebuttal": {
      "room_by_room_analysis": [
        {
          "location": "specific room/elevation",
          "issue": "what was omitted or under-scoped",
          "evidence": "photo/report references",
          "code_standard": "applicable code section with citation",
          "correct_scope": "what should be included",
          "reasoning": "technical/regulatory justification"
        }
      ],
      "code_violations": [
        {
          "code_section": "specific IRC/IBC/NEC section",
          "violation_description": "what code requirement was violated",
          "life_safety_impact": "safety implications",
          "enforcement_consequence": "regulatory implications"
        }
      ],
      "general_conditions_justification": {
        "trade_count": "number of trades requiring coordination",
        "supervision_requirements": "project management needs",
        "site_logistics": "dumpster/toilet/storage necessity",
        "op_calculation": "overhead and profit justification"
      },
      "aesthetic_matching_failures": [
        {
          "system": "affected building system",
          "mismatch_description": "specific matching failure",
          "lkq_violation": "how it violates policy standards",
          "photo_evidence": "supporting visual evidence"
        }
      ]
    },
    "professional_conclusion": {
      "omitted_areas_count": "number of omitted rooms/systems",
      "major_safety_risks": ["list of life-safety concerns"],
      "estimated_value_impact": "financial impact if calculable",
      "formal_demand": "We respectfully request that all omitted items identified in this rebuttal be added to the estimate and paid in full per the policy terms and applicable building codes."
    }
  }
}
 
VALIDATION REQUIREMENTS:
 
Before generating output, verify:
□ All 5 executive summary categories are populated with specific technical details
□ Each summary follows the exact format: "[Category]: [2-3 sentence explanation]"
□ Room-by-room analysis covers all major deficiencies
□ Code violations include specific section references
□ Professional tone maintained throughout
□ All claims supported by evidence references
 
CRITICAL: The executive_summary array must contain exactly 5 strings in the specified format, beginning with the category name and colon, followed by detailed technical and regulatory justification.
 
Start with { and end with }. Include no other content.
"""
        }

In [30]:
import json

def convert_scopes_to_llm_text(input_path="scopes.json", output_path="scopes_for_llm.txt"):
    try:
        with open(input_path, "r") as f:
            scopes = json.load(f)

        output_lines = ["### Available Scopes (from Clear Estimates API)", ""]

        for scope in scopes:
            title = scope.get("title", "Untitled")
            scope_id = scope.get("scope_id", "unknown")
            units = scope.get("units", "unspecified")

            # Concise bullet format
            output_lines.append(f"- **{title}**  \n  `scope_id: {scope_id}`  \n  _Units_: {units}")
            output_lines.append("")  # extra newline for spacing

        # Write to a .txt file
        with open(output_path, "w") as f:
            f.write("\n".join(output_lines))

        print(f"✅ {len(scopes)} scopes written to {output_path}")

    except Exception as e:
        print(f"❌ Error during scope conversion: {e}")


In [31]:
convert_scopes_to_llm_text()

✅ 322 scopes written to scopes_for_llm.txt


In [4]:
def load_llm_scopes_from_file(filepath):
    """Load LLM output from a text file containing JSON array."""
    try:
        with open(filepath, "r") as f:
            # Clean any stray markdown fencing or whitespace if needed
            raw = f.read().strip()
            # In case file is wrapped in ```json ... ```
            if raw.startswith("```json"):
                raw = raw.strip("```json").strip("```")
            data = json.loads(raw)
        print(f"✅ Loaded {len(data)} scope items from {filepath}")
        return data
    except Exception as e:
        print(f"❌ Failed to load file {filepath}: {e}")
        return []

def create_estimate_from_llm_output(llm_output, zipcode="78749", estimate_type=3):
    url = BASE_URL + "estimates/create"
    headers = {"x-api-key": API_KEY}

    estimate_array = [
        {"scope_id": item["scope_id"], "quantity": item["quantity"]}
        for item in llm_output
    ]

    payload = {
        "zipcode": zipcode,
        "estimatearray": estimate_array,
        "type": estimate_type
    }

    response = requests.post(url, headers=headers, json=payload)
    print("ESTIMATE REQUEST:", response.status_code)

    try:
        response_data = response.json()
        print(json.dumps(response_data, indent=2))
        return response_data
    except json.JSONDecodeError:
        print("Invalid JSON in response")
        return None


In [5]:
import json
import requests
import os
# ---------- Async Gemini Runner ----------

async def run_block(label, prompt, file_parts=None):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    if file_parts:
        contents[0].parts.extend(file_parts)

    output = ""
    try:
        print(f"🔹 Running {label}...")
        stream = client.models.generate_content_stream(
            model=model_name,
            contents=contents,
            config=generate_content_config
        )
        for chunk in stream:  # ✅ DO NOT use 'await'
            output += chunk.text
        print(f"✅ {label} complete ({len(output)} chars)")
    except Exception as e:
        output = f"[ERROR in {label}] {e}"
        print(output)

    return label, output




# ---------- Master Pipeline ----------

async def run_aistimate_pipeline(file_paths,pricing_path):
    prompts = build_prompts()

    # Assign files
    # 1) Carrier: only the first file
    carrier_parts = [make_part(file_paths[0])]

    # 2) Evidence: file_paths[0] plus file_paths[2:]
    evidence_paths = [file_paths[0]] + file_paths[2:]
    evidence_parts = [make_part(path) for path in evidence_paths]

    # 3) Policy: again, just the first file (if that’s what you meant)
    policy_parts = [make_part(path) for path in file_paths[1:2]]


    # Stage 1: Run code lookup & damage analysis in parallel
    stage1_tasks = [
        run_block("CODE_LOOKUP", prompts["CODE_LOOKUP"], carrier_parts),
        run_block("REPORT_ANALYSIS", prompts["REPORT_ANALYSIS"], evidence_parts),
    ]
    stage1_results = await asyncio.gather(*stage1_tasks)
    context = {label: output for label, output in stage1_results}

    for label, content in stage1_results:
        save_output(label, content)

    # Stage 2: Scoping logic (needs prior outputs)
    scoping_context = (
        f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
    )
    label, scoping_output = await run_block("SCOPING_LOGIC", prompts["SCOPING_LOGIC"] + "nn" + scoping_context)
    save_output(label, scoping_output)
    context["SCOPING_LOGIC"] = scoping_output

    pricing_context = (
        f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
        f"--- SCOPING LOGIC ---n{context['SCOPING_LOGIC']}"
    )
    
    label, pricing_output = await run_block("PRICING_LOGIC", prompts["PRICING_LOGIC"] + "nn" + pricing_context,policy_parts)
    save_output(label, pricing_output)
    filepath = os.path.join(pricing_path,"output_pricing_logic.txt")
    print("Pricing output saved to:", filepath)
    scope_data = load_llm_scopes_from_file(filepath)
    if scope_data:
    # Create estimate
        valid_scopes = [
        item for item in scope_data
        if not item["scope_id"].startswith("MISSING_")
        ]

# Call your estimate creation function
        estimate_response = create_estimate_from_llm_output(valid_scopes, zipcode="80210", estimate_type=2)
        # print("Estimate response:", estimate_response)
        # Save estimate_response to file in pricing_path directory
        estimate_filepath = os.path.join(pricing_path, "estimate_response.json")
        with open(estimate_filepath, "w") as f:
            f.write(json.dumps(estimate_response, indent=2))
        print("Estimate response saved to:", estimate_filepath)
        
    
    



# Use that structured version in the context
    estimate_context = (
    f"--- CODE MANDATES ---\n{context['CODE_LOOKUP']}\n\n"
    f"--- DAMAGE FINDINGS ---\n{context['REPORT_ANALYSIS']}\n\n" 
    f"--- SCOPING LOGIC ---\n{context['SCOPING_LOGIC']}\n\n"   
    f"--- PRICING LOGIC ---\n{pricing_output}\n\n"
    f"--- SELECTED SCOPES ---\n{estimate_response}\n"
    )   

        # f"--- POLICY LOGIC ---n{context['POLICY_LOGIC_output']}nn"
    
    label, estimate_output = await run_block("ESTIMATE", prompts["ESTIMATE"] + "nn" + estimate_context)
    save_output(label, estimate_output)

    # Stage 4: Rebuttal
    # Stage 4: Rebuttal (pass carrier file for comparison)
    label, rebuttal_output = await run_block(
    "REBUTTAL",
    prompts["REBUTTAL"] + "nn" + estimate_output,
    file_parts=carrier_parts  # 🔹 passes carrier estimate as input context
    )
    save_output(label, rebuttal_output)


    return {
        "code_lookup": context["CODE_LOOKUP"],
        "report_analysis": context["REPORT_ANALYSIS"],
        "scoping_logic": context["SCOPING_LOGIC"],
        "estimate_output": estimate_output,
        "rebuttal_output": rebuttal_output,
    }


In [6]:
API_KEY = "emym6vnmxyo0d62vz3w1kaj"

# Step 2: Set the base URL (use sandbox for testing)
BASE_URL = "https://api.clearestimates.com/cepia/"

In [9]:
import requests
import json
filepath = "outputs/Jul25/2500095/run4/output_scoping_logic.txt"
scope_data = load_llm_scopes_from_file(filepath)
print("Loaded scopes:", scope_data)
if scope_data:
    # Create estimate
    estimate_response = create_estimate_from_llm_output(scope_data, zipcode="80210", estimate_type=2)

    if estimate_response:
        # Save to file
        with open("estimate.txt", "w") as f:
            f.write(json.dumps(estimate_response, indent=2))
        
    else:
        print("❌ Estimate creation returned no result.")
else:
    print("❌ No scope data loaded; cannot proceed with estimate creation.")



✅ Loaded 8 scope items from outputs/Jul25/2500095/run4/output_scoping_logic.txt
Loaded scopes: [{'scope_id': 'a84dc25a-485b-4f08-8470-8e95c943019d', 'quantity': 2700, 'justification': 'Widespread hail damage documented across all slopes in annotated photos 1-50 requires full replacement. This scope includes tear-off of existing shingles, installation of new code-compliant underlayment (per IRC R905.1.1), starter strips, and dimensional shingles to restore the primary water-shedding surface. Quantity is based on an estimated 2200 sqft home footprint with a 1.2 pitch/waste factor.'}, {'scope_id': 'ceedede7-b7b0-44ac-892c-64da2e179538', 'quantity': 9, 'justification': 'IRC R905.1 mandates deck inspection upon shingle removal. This is an allowance for 9 sheets (approx. 10% of roof area) to repair any damaged, delaminated, or improperly fastened decking found during inspection, ensuring a solid substrate for the new roof system.'}, {'scope_id': 'c4f9b3f8-6256-47dc-bc1f-baad1ea80d24', 'quant

In [ ]:
output_dir = f"outputs/Jul25/2550002/run1"
pricing_path = output_dir
def save_output(label: str, content: str):
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "2550002/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "scopes_for_llm.txt",
    "2550002/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_3A97D8FE-84D6-4BFA-91AD-39F43E790DC5.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_72CE907A-5EDA-42FC-A9C2-3D4245CC834E.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_093D1853-BBE9-47A7-86EF-1769895C5FBE.jpeg",
    "2550002/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_459D33C9-7EA8-451F-A4EE-1C09493F9BEA.jpeg"
],pricing_path)

FileNotFoundError: [Errno 2] No such file or directory: '2550002/2550002 Carrier Estimate Insurance Carrier Estimate.pdf'

In [12]:
output_dir = f"outputs/Jul31/2500095/run7"
def save_output(label: str, content: str):
   
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/117/ai estimate useful docs/2500095_Correspondence_Correspondence from IC_From_Nationwide Mutual Insurance Company_Supplement Estimate 2.pdf",
    "scopes_for_llm.txt",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/117/ai estimate useful docs/Binder1 Photos_ Arnold, Marc.pdf",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/117/ai estimate useful docs/LN Re-inspect Report.pdf"
],output_dir)

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (6786 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (14278 chars)
📝 Saved: outputs/Jul31/2500095/run7/output_code_lookup.txt
📝 Saved: outputs/Jul31/2500095/run7/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (11949 chars)
📝 Saved: outputs/Jul31/2500095/run7/output_scoping_logic.txt
🔹 Running PRICING_LOGIC...
✅ PRICING_LOGIC complete (7328 chars)
📝 Saved: outputs/Jul31/2500095/run7/output_pricing_logic.txt
Pricing output saved to: outputs/Jul31/2500095/run7/output_pricing_logic.txt
✅ Loaded 17 scope items from outputs/Jul31/2500095/run7/output_pricing_logic.txt
ESTIMATE REQUEST: 200
{
  "estimate_id": 3604,
  "estimates": [
    {
      "title": "Roofing Replacement  - Dimensional",
      "description": "Demolish existing layer of roofing and replace with architectural/laminate/dimensional 240lb, 30 year fiberglass shingles. Flash chimney, plumbing vents, and valleys with aluminum. Continuous

In [18]:
output_dir = f"outputs/Jul31/2500074/run6"
def save_output(label: str, content: str):
   
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Carrier Estimate ($16,113.56) Insurance Carrier Estimate.pdf",
    "scopes_for_llm.txt",
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Eagleview Report - Grace Forensic Plaintiff Expert Estimate.PDF",
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Forensic Damage Assessment - Grace Forensic Plaintiff Expert Estimate.pdf",
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Inspection Report For Primary Structure 04-16-2025 Plaintiff Expert Estimate_compressed.pdf",
],output_dir)

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (7897 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (26782 chars)
📝 Saved: outputs/Jul31/2500074/run6/output_code_lookup.txt
📝 Saved: outputs/Jul31/2500074/run6/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (14479 chars)
📝 Saved: outputs/Jul31/2500074/run6/output_scoping_logic.txt
🔹 Running PRICING_LOGIC...
✅ PRICING_LOGIC complete (12622 chars)
📝 Saved: outputs/Jul31/2500074/run6/output_pricing_logic.txt
Pricing output saved to: outputs/Jul31/2500074/run6/output_pricing_logic.txt
✅ Loaded 27 scope items from outputs/Jul31/2500074/run6/output_pricing_logic.txt
ESTIMATE REQUEST: 200
{
  "estimate_id": 3643,
  "estimates": [
    {
      "title": "Roofing Replacement  - Dimensional",
      "description": "Demolish existing layer of roofing and replace with architectural/laminate/dimensional 240lb, 30 year fiberglass shingles. Flash chimney, plumbing vents, and valleys with aluminum. Continuou

In [19]:
def save_output(label: str, content: str):
    output_dir = f"outputs/PRICE_CASE2/RUN1"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_3A97D8FE-84D6-4BFA-91AD-39F43E790DC5.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_093D1853-BBE9-47A7-86EF-1769895C5FBE.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_72CE907A-5EDA-42FC-A9C2-3D4245CC834E.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_459D33C9-7EA8-451F-A4EE-1C09493F9BEA.jpeg",
], output_dir)

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (8411 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (15629 chars)
📝 Saved: outputs/PRICE_CASE2/RUN1/output_code_lookup.txt
📝 Saved: outputs/PRICE_CASE2/RUN1/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (11813 chars)
📝 Saved: outputs/PRICE_CASE2/RUN1/output_scoping_logic.txt
🔹 Running PRICING_LOGIC...
✅ PRICING_LOGIC complete (8200 chars)
📝 Saved: outputs/PRICE_CASE2/RUN1/output_pricing_logic.txt
Pricing output saved to: outputs/Jul31/2500074/run6/output_pricing_logic.txt
✅ Loaded 27 scope items from outputs/Jul31/2500074/run6/output_pricing_logic.txt
ESTIMATE REQUEST: 200
{
  "estimate_id": 3654,
  "estimates": [
    {
      "title": "Roofing Replacement  - Dimensional",
      "description": "Demolish existing layer of roofing and replace with architectural/laminate/dimensional 240lb, 30 year fiberglass shingles. Flash chimney, plumbing vents, and valleys with aluminum. Continuous ridge v